# RL Vectorizer — Colab SFT Training

In [ ]:
!git clone https://github.com/dersteppenwolfruowen-316/RL-Vec.git
%cd /content/RL-Vec

# 核心依赖（不强制 numpy 版本，避免冲突）
!pip install -q transformers peft accelerate bitsandbytes datasets
!pip install -q lxml cairosvg pillow scikit-image opencv-python tensorboard
!pip install -q hydra-core omegaconf shapely
!pip uninstall -y torchao

# numpy<2 放在最后单独降级（避免 pip resolve 时与其他包冲突）
!pip install -q "numpy<2" --force-reinstall 2>/dev/null || true

import torch
print(f"Torch CUDA: {torch.cuda.is_available()}")
print(f"PyTorch: {torch.__version__}")
print('Cell 1 done')

In [ ]:
import urllib.request, zipfile, os

DATA_DIR = "data/resplan"
os.makedirs(DATA_DIR, exist_ok=True)
ZIP = os.path.join(DATA_DIR, "ResPlan.zip")
PKL = os.path.join(DATA_DIR, "ResPlan.pkl")

if not os.path.exists(ZIP):
    print("Downloading ResPlan.zip (96MB)...")
    urllib.request.urlretrieve(
        "https://github.com/m-agour/ResPlan/releases/download/1.0.0/ResPlan.zip", ZIP)

if not os.path.exists(PKL):
    print("Extracting...")
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(DATA_DIR)

if not os.path.exists(os.path.join(DATA_DIR, "metadata.jsonl")):
    !python convert_resplan.py

if not os.path.exists(os.path.join(DATA_DIR, "sft_train.jsonl")):
    !python scripts/prepare_sft_data.py

svg_count = len(os.listdir(os.path.join(DATA_DIR, "svgs"))) if os.path.exists(os.path.join(DATA_DIR, "svgs")) else 0
png_count = len(os.listdir(os.path.join(DATA_DIR, "bitmaps"))) if os.path.exists(os.path.join(DATA_DIR, "bitmaps")) else 0
sft_count = sum(1 for _ in open(os.path.join(DATA_DIR, "sft_train.jsonl"))) if os.path.exists(os.path.join(DATA_DIR, "sft_train.jsonl")) else 0
print(f"Data ready: {png_count} bitmaps, {sft_count} SFT samples")
print('Cell 2 done')

In [ ]:
%cd /content/RL-Vec
!python scripts/train_sft.py \
    --max-samples 100 \
    --epochs 3 \
    --lr 1e-4 \
    --lora-rank 8 \
    --lora-alpha 16 \
    --grad-accum-steps 8 \
    --quantization 4bit \
    --save-dir checkpoints/sft \
    --log-interval 5

In [ ]:
!pip install -q "numpy<2" scikit-image

import sys, os, json, re, io
import numpy as np
from PIL import Image
from lxml import etree
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

TEST_IDS = ["resplan_00013", "resplan_00017", "resplan_00021",
            "resplan_00025", "resplan_00033"]
DATA_DIR = "data/resplan"
SFT_CKPT = "checkpoints/sft/final"
MODEL_NAME = "Qwen/Qwen2.5-VL-3B-Instruct"
PROMPT = "Convert this architectural floor plan to SVG. First analyze its structure, then generate the SVG step by step."

if not torch.cuda.is_available():
    print("CUDA not available")
else:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                 bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4")

    def load_img(sid):
        return Image.open(f"{DATA_DIR}/bitmaps/{sid}.png").convert("RGB")

    def extract_svg(text):
        for p in [r"<svg_output>\s*(.*?)\s*</svg_output>", r"```(?:svg)?\s*\n?(.*?)\n?\s*```", r"(<svg[\s\S]*?</svg>)"]:
            m = re.search(p, text, re.DOTALL)
            if m: return m.group(1).strip()
        return text.strip()

    def valid_svg(code):
        try: return etree.fromstring(code.encode()).tag.endswith("svg")
        except: return False

    def count_elems(code):
        try:
            r = etree.fromstring(code.encode())
            return len(r.xpath("//svg:path|//svg:line|//svg:rect", namespaces={"svg": "http://www.w3.org/2000/svg"}))
        except: return 0

    def fmt_score(text):
        return sum(1 for t in ["<analysis>", "<outer_wall>", "<svg_output>"] if t in text)

    def infer(model, processor, image):
        msgs = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": PROMPT]}]
        txt = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp = processor(text=[txt], images=[image], return_tensors="pt")
        inp = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inp.items()}
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=1024, do_sample=False, temperature=0.6, top_p=0.95)
        return processor.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

    results = []
    processor = AutoProcessor.from_pretrained(MODEL_NAME, use_fast=False)

    print("=" * 60)
    print("Zero-shot")
    print("=" * 60)
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME, quantization_config=quant, device_map="auto", torch_dtype=torch.float16)
    for sid in TEST_IDS:
        img = load_img(sid)
        resp = infer(model, processor, img)
        svg = extract_svg(resp)
        ok = valid_svg(svg)
        results.append({"id": sid, "model": "zero-shot", "valid": ok,
                        "elems": count_elems(svg) if ok else 0, "fmt": fmt_score(resp)})
        print(f"  {'ok' if ok else 'X'} {sid}: elems={results[-1]['elems']} fmt={results[-1]['fmt']}/3")
    del model; torch.cuda.empty_cache()

    print()
    print("=" * 60)
    print(f"SFT ({SFT_CKPT})")
    print("=" * 60)
    if os.path.exists(SFT_CKPT):
        model = Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME, quantization_config=quant, device_map="auto", torch_dtype=torch.float16)
        model = PeftModel.from_pretrained(model, SFT_CKPT)
        for sid in TEST_IDS:
            img = load_img(sid)
            resp = infer(model, processor, img)
            svg = extract_svg(resp)
            ok = valid_svg(svg)
            results.append({"id": sid, "model": "sft", "valid": ok,
                            "elems": count_elems(svg) if ok else 0, "fmt": fmt_score(resp)})
            print(f"  {'ok' if ok else 'X'} {sid}: elems={results[-1]['elems']} fmt={results[-1]['fmt']}/3")
        del model; torch.cuda.empty_cache()
    else:
        print("  (SFT checkpoint not found)")

    print()
    print("=" * 68)
    print("  SFT vs Zero-shot")
    print("=" * 68)
    by_id = {}
    for r in results:
        by_id.setdefault(r["id"], {})[r["model"]] = r
    for sid in TEST_IDS:
        z = by_id.get(sid, {}).get("zero-shot", {})
        s = by_id.get(sid, {}).get("sft", {})
        if not z and not s: continue
        print(f"  {sid:<14} Valid    {str(z.get('valid', '')):<12} {str(s.get('valid', '')):<12}")
        print(f"  {'':14} Elements {str(z.get('elems', 0)):<12} {str(s.get('elems', 0)):<12}")
        print(f"  {'':14} Fmt/3    {str(z.get('fmt', 0)):<12} {str(s.get('fmt', 0)):<12}")
        print()
    n_z = sum(1 for r in results if r["model"] == "zero-shot" and r["valid"])
    n_s = sum(1 for r in results if r["model"] == "sft" and r["valid"])
    print(f"  SVG Valid: zero-shot={n_z}/{len(TEST_IDS)}  SFT={n_s}/{len(TEST_IDS)}")
    print("=" * 68)
    with open("eval_results.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"Saved to eval_results.json")